In [1]:
import pandas as pd
import re

In [2]:
http_dataset=pd.read_csv(r'E:\UNMC Internship Document\Github Upload\data\http_dataset.csv')

In [6]:
http_filtered_df=pd.DataFrame(columns=['url', 'label'])

for i in http_dataset.index:
    if http_dataset.loc[i,'https_token']==1 and http_dataset.loc[i,'status']=='legitimate':
        http_filtered_df.loc[len(http_filtered_df),'url']=http_dataset.loc[i,'url']
        http_filtered_df.loc[len(http_filtered_df)-1,'label']=http_dataset.loc[i,'status']

http_filtered_df


,url,label
0,http://www.crestonwood.com/router.php,legitimate
1,http://rgipt.ac.in,legitimate
2,http://www.iracing.com/tracks/gateway-motorspo...,legitimate
3,http://www.mutuo.it,legitimate
4,http://vamoaestudiarmedicina.blogspot.com/,legitimate
...,...,...
3167,http://www.siteglimpse.com/automationalley.com,legitimate
3168,http://sheetdownload.com/,legitimate
3169,http://www.answers.com/Q/What_are_the_sizes_of...,legitimate
3170,http://www.fontspace.com/category/blackletter,legitimate


In [8]:
http_filtered_df.to_csv('legitimate_http_sites_dataset_v2.csv')

In [10]:
"""
find_live_http_sites.py
------------------------
Filters a list of candidate http:// URLs down to the ones that are BOTH
  (1) live (respond without error), and
  (2) genuinely served over HTTP -- i.e. they do NOT redirect to https://

These are the only valid `has_https = 0` legitimate examples: a site that
redirects http -> https is really an HTTPS site and must not be labelled http.

Run this on your own machine (it needs open internet access).

    python find_live_http_sites.py

Requirements:  pip install requests pandas
"""

import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---- Config ----
INPUT_CSV   = r"E:\UNMC Internship Document\Github Upload\notebooks\legitimate_http_sites_dataset_v2.csv"   # your 3172 candidates
OUTPUT_CSV  = "legit_http_sites_LIVE_v3.csv"                     # verified survivors
TIMEOUT     = 8        # seconds per request
MAX_WORKERS = 40       # parallel requests (lower if your network complains)
TARGET      = 60       # stop-early goal (set to None to check ALL candidates)

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/124.0 Safari/537.36")
}


def check(url):
    """Return a result dict if the URL is live AND stays on http, else None."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT,
                         allow_redirects=True)
    except requests.RequestException:
        return None

    final = r.url                      # URL after any redirects
    live  = r.status_code < 400        # 2xx / 3xx-resolved = usable page

    if live and final.lower().startswith("http://"):
        return {"url": url,            # keep the ORIGINAL http url
                "label": "legitimate",
                "final_url": final,
                "status": r.status_code}
    return None


def main():
    df = pd.read_csv(INPUT_CSV)
    urls = df["url"].astype(str).tolist()
    print(f"Loaded {len(urls)} candidate http:// URLs")

    survivors, checked = [], 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(check, u): u for u in urls}
        for fut in as_completed(futures):
            checked += 1
            res = fut.result()
            if res:
                survivors.append(res)
                print(f"  [{len(survivors):>3}] LIVE HTTP  {res['url']}")

            if checked % 200 == 0:
                print(f"...checked {checked}/{len(urls)}  "
                      f"| found {len(survivors)} genuine http")

            if TARGET and len(survivors) >= TARGET:
                print(f"\nReached target of {TARGET} live http sites - stopping early.")
                break

    out = pd.DataFrame(survivors)
    out.to_csv(OUTPUT_CSV, index=False)
    print(f"\nDone. {len(out)} genuine live-http legit sites -> {OUTPUT_CSV}")
    print("Columns: url, label, final_url, status")
    print("\nNext: eyeball the survivors, then append the [url, label] columns "
          "to dataset_live_links_v2.csv (label them 0 = legitimate).")


if __name__ == "__main__":
    main()

Loaded 3172 candidate http:// URLs
  [  1] LIVE HTTP  http://press-preview.weebly.com
  [  2] LIVE HTTP  http://www.crestonwood.com/router.php
  [  3] LIVE HTTP  http://www.enkiquotes.com/
  [  4] LIVE HTTP  http://vamoaestudiarmedicina.blogspot.com/
  [  5] LIVE HTTP  http://www.routeralley.com/guides/nat.pdf
  [  6] LIVE HTTP  http://www.abbreviations.com/term/16642
  [  7] LIVE HTTP  http://careers.stateuniversity.com/pages/100000947/Search-Engines-Future-User-Preferences-Artificial-Intelligence.html
  [  8] LIVE HTTP  http://www.novell.com/documentation/oes11/dsfw_bp_lx/
  [  9] LIVE HTTP  http://toppave.com.sg
  [ 10] LIVE HTTP  http://www.biology4kids.com/files/cell2_activetran.html
  [ 11] LIVE HTTP  http://www.piano-play-it.com/layout-piano-keys.html
  [ 12] LIVE HTTP  http://www.gaudi.ch/GaudiLabs/?page_id=19
  [ 13] LIVE HTTP  http://www.definitions.net/definition/compact%20disc
  [ 14] LIVE HTTP  http://www.codeboiler.net/
  [ 15] LIVE HTTP  http://www.definitions.net/defini